# Navier–Stokes

**DD2365 Advanced Computation in Fluid Mechanics**
KTH Royal Institute of Technology, Stockholm, Sweden

*FEniCSx 0.11 port of the legacy FEniCS Navier-Stokes notebook.*

---

Copyright (C) 2020–2026 Johan Hoffman (johanhoffman@me.com).
This is free software under the GNU LGPL v3 or later.

In [ ]:
# This program is an example file for the course
# DD2365 Advanced Computation in Fluid Mechanics,
# KTH Royal Institute of Technology, Stockholm, Sweden.

# Copyright (C) 2020-2026 Johan Hoffman (johanhoffman@me.com)

# This file is part of the course DD2365 Advanced Computation in Fluid Mechanics
# KTH Royal Institute of Technology, Stockholm, Sweden
#
# This is free software: you can redistribute it and/or modify
# it under the terms of the GNU Lesser General Public License as published by
# the Free Software Foundation, either version 3 of the License, or
# (at your option) any later version.

# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
import numpy as np
import time
import ufl
import basix.ufl as bufl
from mpi4py import MPI
from matplotlib import pyplot as plt

import dolfinx
import dolfinx.cpp
from dolfinx import fem
from dolfinx.fem import functionspace, Function, form, assemble_scalar, Constant
from dolfinx.fem.petsc import (
    assemble_matrix, assemble_vector, apply_lifting, set_bc,
    create_matrix, create_vector,
)
from petsc4py import PETSc

In [ ]:
# --- canonical: gmsh_rect_minus_circles v2 ---
import math
import gmsh
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio


def gmsh_rect_minus_circles(L, H, circles, resolution):
    """Rectangle [0,L]×[0,H] minus circular holes, meshed with gmsh OCC.

    Parameters
    ----------
    L, H : float
        Rectangle dimensions.
    circles : list of (cx, cy, r)
        Circle centres and radii to subtract.
    resolution : int
        Mesh density parameter.  The global size bound is

            lc = 0.65 * sqrt(L² + H²) / resolution

        This matches the mshr/CGAL semantics used in the legacy FEniCS
        notebooks: mshr's ``resolution`` sets a CGAL size bound equal to
        the bounding-box diagonal divided by ``resolution``; gmsh realised
        edges are approximately 0.6× that bound.  The factor 0.65 was
        calibrated so that the standard test case (L=4, H=2, 3 circular
        holes, resolution=32) produces ≈2319 cells — matching the legacy
        mshr mesh (2319 cells, 1247 P1 dofs).

    Returns
    -------
    msh : dolfinx.mesh.Mesh
    cell_tags : dolfinx.mesh.MeshTags   (fluid domain tag = 10)

    Note
    ----
    Facet tags are not produced here.  Call ``tag_boundaries(msh, L, H)``
    on the final mesh (after any refinement) to obtain boundary MeshTags.
    """
    _ALPHA = 0.65
    lc = _ALPHA * math.sqrt(L**2 + H**2) / resolution

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    rect = gmsh.model.occ.addRectangle(0.0, 0.0, 0.0, L, H)
    disks = [(2, gmsh.model.occ.addDisk(cx, cy, 0.0, r, r)) for cx, cy, r in circles]
    if disks:
        gmsh.model.occ.cut([(2, rect)], disks)
    gmsh.model.occ.synchronize()

    gmsh.option.setNumber("Mesh.MeshSizeMax", lc)

    surfaces = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(2, [s[1] for s in surfaces], tag=10, name="domain")

    gmsh.model.mesh.generate(2)

    mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, rank=0, gdim=2)
    gmsh.finalize()
    return mesh_data.mesh, mesh_data.cell_tags
# --- end canonical: gmsh_rect_minus_circles ---

In [ ]:
# --- canonical: refine_cells v2 ---
import numpy as np
import dolfinx.mesh
from dolfinx.mesh import RefinementOption


def refine_cells(msh, predicate_or_mask):
    """Refine selected cells using the Plaza algorithm.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    predicate_or_mask : callable or array-like of bool
        Either a callable ``f(midpoints) -> bool array`` where
        ``midpoints`` has shape ``(ncells, gdim)``, or a boolean array
        of length ``num_local_cells`` that directly marks which cells
        to refine (useful when marks come from an existing DG0 field).

    Returns
    -------
    refined_msh : dolfinx.mesh.Mesh
    parent_cells : np.ndarray[np.int32]  shape (num_refined_cells,)
    parent_facets : np.ndarray[np.int8]  shape (num_refined_cells,)
        Local parent-facet index per refined cell, or -1 for interior.
    """
    tdim = msh.topology.dim
    num_cells = msh.topology.index_map(tdim).size_local

    if callable(predicate_or_mask):
        msh.topology.create_entities(1)
        msh.topology.create_connectivity(tdim, 0)
        midpoints = dolfinx.mesh.compute_midpoints(
            msh, tdim, np.arange(num_cells, dtype=np.int32)
        )
        marked = np.where(predicate_or_mask(midpoints))[0].astype(np.int32)
    else:
        mask = np.asarray(predicate_or_mask, dtype=bool)
        marked = np.where(mask[:num_cells])[0].astype(np.int32)

    if len(marked) == 0:
        n = msh.topology.index_map(tdim).size_local
        return msh, np.arange(n, dtype=np.int32), np.full(n, -1, dtype=np.int8)

    msh.topology.create_entities(1)
    msh.topology.create_connectivity(tdim, 1)
    edges = dolfinx.mesh.compute_incident_entities(msh.topology, marked, tdim, 1)
    edges = np.unique(edges).astype(np.int32)
    return dolfinx.mesh.refine(
        msh, edges, option=RefinementOption.parent_cell_and_facet
    )
# --- end canonical: refine_cells ---

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]×[0,H] rectangle with holes.

    Assigns integer tags to all exterior boundary facets:

        left=1  (x ≈ 0),   right=2 (x ≈ L),
        lower=3 (y ≈ 0),   upper=4 (y ≈ H),
        objects=5 (remaining exterior facets — circle boundaries).

    Corners are unambiguous: boundary facets are edges, not vertices, so a
    corner vertex is shared by one vertical and one horizontal edge — each
    edge belongs to exactly one group and no facet is tagged twice.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    L, H : float
        Rectangle dimensions.
    eps : float, optional
        Coordinate tolerance.  Default ``1e-6 * max(L, H)``.

    Returns
    -------
    dolfinx.mesh.MeshTags
        Must be recomputed whenever the mesh changes (e.g. after refinement).
    """
    if eps is None:
        eps = 1e-6 * max(L, H)

    fdim = msh.topology.dim - 1
    msh.topology.create_entities(fdim)
    msh.topology.create_connectivity(fdim, msh.topology.dim)

    left  = locate_entities_boundary(msh, fdim, lambda x: x[0] <= eps)
    right = locate_entities_boundary(msh, fdim, lambda x: x[0] >= L - eps)
    lower = locate_entities_boundary(msh, fdim, lambda x: x[1] <= eps)
    upper = locate_entities_boundary(msh, fdim, lambda x: x[1] >= H - eps)

    known = np.unique(np.concatenate([left, right, lower, upper]))
    all_bdry = locate_entities_boundary(
        msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool)
    )
    objects = np.setdiff1d(all_bdry, known)

    indices = np.concatenate([left, right, lower, upper, objects]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(left),    1, dtype=np.int32),
        np.full(len(right),   2, dtype=np.int32),
        np.full(len(lower),   3, dtype=np.int32),
        np.full(len(upper),   4, dtype=np.int32),
        np.full(len(objects), 5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])
# --- end canonical: tag_boundaries ---

In [ ]:
# --- canonical: plot_helpers v2 ---
import numpy as np
import matplotlib.pyplot as plt
import basix.ufl as _bufl
from dolfinx.fem import functionspace, Function


def plot_mesh(msh, title="Mesh"):
    """Plot a 2-D triangular mesh using matplotlib triplot."""
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _p1_scalar_space(msh):
    return functionspace(msh, ("Lagrange", 1))


def _scatter_p1_scalar(f1):
    """Scatter a P1 scalar Function values to geometry nodes.

    Returns (x, geometry_connectivity, node_values).
    """
    V = f1.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = f1.x.array.real[ldm.ravel()]
    return x, gdm, vals


def plot_p1(u, title="Solution"):
    """Plot a P1 scalar Function using matplotlib tripcolor (Gouraud shading)."""
    plot_scalar(u, title=title)


def plot_scalar(f, title="Scalar field"):
    """Plot any scalar Lagrange Function via tripcolor (interpolates to P1 if needed)."""
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    if el.degree == 1 and el.reference_value_shape == ():
        f1 = f
    else:
        f1 = Function(_p1_scalar_space(msh))
        f1.interpolate(f)
    x, gdm, vals = _scatter_p1_scalar(f1)
    fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def plot_vector(u, title="Vector field", quiver=True, quiver_stride=8):
    """Plot a 2-D vector Lagrange Function as colour map of |u| + optional quiver.

    Parameters
    ----------
    u : dolfinx.fem.Function   value shape (2,); any Lagrange degree
    quiver : bool              overlay subsampled arrows (default True)
    quiver_stride : int        take every N-th mesh node for arrows
    """
    msh = u.function_space.mesh
    gdim = msh.geometry.dim
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    npts = x.shape[0]

    # Interpolate to P1 vector so values align with geometry nodes
    V1v = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=(gdim,)))
    u1v = Function(V1v)
    u1v.interpolate(u)

    # Scatter: for a block-size-2 P1 space, dofmap.list gives block indices
    ldm = V1v.dofmap.list      # (ncells, 3)  — block indices
    arr = u1v.x.array.real     # length npts * gdim, interleaved per block
    ux = np.zeros(npts)
    uy = np.zeros(npts)
    ux[gdm.ravel()] = arr[ldm.ravel() * gdim + 0]
    uy[gdm.ravel()] = arr[ldm.ravel() * gdim + 1]
    mag = np.sqrt(ux**2 + uy**2)

    fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, mag, shading="gouraud", cmap="viridis")
    plt.colorbar(tc, ax=ax, label="|u|")
    if quiver:
        idx = np.arange(0, npts, quiver_stride)
        ax.quiver(x[idx, 0], x[idx, 1], ux[idx], uy[idx],
                  color="white", alpha=0.6, width=0.002, scale_units="xy", scale=2.0)
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
# --- end canonical: plot_helpers ---

In [ ]:
# --- canonical: xdmf_series v1 ---
import sys
import pathlib
import basix.ufl as _bufl
from mpi4py import MPI
from dolfinx.io import XDMFFile
from dolfinx.fem import functionspace, Function


def _to_p1(f):
    """Interpolate f to P1 (scalar or vector) if needed."""
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    vshape = el.reference_value_shape
    if el.degree == 1:
        return f
    if vshape == ():
        V1 = functionspace(msh, ("Lagrange", 1))
    else:
        V1 = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=vshape))
    f1 = Function(V1, name=f.name)
    f1.interpolate(f)
    return f1


class XDMFSeries:
    """Time-series XDMF writer.

    Opens one XDMFFile, writes the mesh once on the first write() call,
    then appends each time step.  Higher-degree functions are automatically
    interpolated to P1 before writing.

    Usage::

        xdmf = XDMFSeries("output/solution.xdmf")
        for t in ...:
            xdmf.write([u1, p1, w1], t)
        xdmf.close()          # or: `with XDMFSeries(...) as xdmf:`

    Parameters
    ----------
    path : str or Path
        Output ``.xdmf`` file path.  The ``.h5`` sidecar is written alongside.
    download : bool, optional
        If True *and* running on Colab, tar the pair and trigger a download
        when :meth:`close` is called.
    """

    def __init__(self, path, download=False):
        self._path = pathlib.Path(path)
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self._download = download
        self._xdmf = XDMFFile(MPI.COMM_WORLD, str(self._path), "w")
        self._mesh_written = False

    def write(self, funcs, t):
        """Write *funcs* at time *t*."""
        funcs_p1 = [_to_p1(f) for f in funcs]
        if not self._mesh_written:
            self._xdmf.write_mesh(funcs_p1[0].function_space.mesh)
            self._mesh_written = True
        for f in funcs_p1:
            self._xdmf.write_function(f, t)

    def close(self):
        """Close the file and optionally trigger a Colab download."""
        self._xdmf.close()
        if self._download and "google.colab" in sys.modules:
            import tarfile
            from google.colab import files as colab_files
            tar_path = str(self._path.with_suffix(".tar.gz"))
            with tarfile.open(tar_path, "w:gz") as tar:
                tar.add(str(self._path), arcname=self._path.name)
                h5 = self._path.with_suffix(".h5")
                if h5.exists():
                    tar.add(str(h5), arcname=h5.name)
            colab_files.download(tar_path)

    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.close()
# --- end canonical: xdmf_series ---

## Problem parameters

Standard benchmark: rectangular channel $[0,4]\times[0,2]$ with one circular
obstacle at $(1, 1)$ of radius $0.2$.  Slip walls (top/bottom), inflow
$u_x=1$ (left), zero-pressure outflow (right).  Re $= U\cdot D / \nu = 100$.

In [ ]:
# Geometry
L = 4.0; H = 2.0
cx, cy, cr = 1.0, 0.5 * H, 0.2
resolution = 32; no_levels = 0

# Physics
nu = 4.0e-3; uin = 1.0

# Solver
num_nnlin_iter = 5

# Time stepping
T = 30.0; plot_freq = 10

## Method

### Governing equations

The incompressible Navier–Stokes equations:

$$
\frac{\partial u}{\partial t} + (\nabla u)\,u + \nabla p - \nu\,\Delta u = 0, \quad \nabla\cdot u = 0
$$

### Discretization

$P_1/P_1$ stabilized FEM with GLS stabilization parameter

$$
\delta_1 = \frac{1}{\sqrt{(1/\Delta t)^2 + (|u|/h)^2}}
$$

and bulk viscosity stabilization $\delta_2 = h\,|u|$ for the divergence-free
constraint.  Trapezoidal (Crank–Nicolson) time stepping with $\bar{u} = \tfrac12(u^n + u^{n-1})$.
The convective term is linearized using the previous nonlinear iterate $u_1$.

In [ ]:
msh, cell_tags = gmsh_rect_minus_circles(L, H, [(cx, cy, cr)], resolution)

for _ in range(no_levels):
    msh, _, _ = refine_cells(
        msh,
        lambda mp: np.sqrt((mp[:, 0] - cx)**2 + (mp[:, 1] - cy)**2) < 1.0
    )

facet_tags = tag_boundaries(msh, L, H)

tdim = msh.topology.dim
fdim = tdim - 1
n_cells = msh.topology.index_map(tdim).size_local
print(f"Mesh: {n_cells} cells")

plot_mesh(msh, title=f"Mesh: {n_cells} cells")

In [ ]:
h_arr = dolfinx.cpp.mesh.h(msh._cpp_object, tdim,
                            np.arange(n_cells, dtype=np.int32))
hmin = float(h_arr.min())
dt = 0.5 * hmin
print(f"hmin = {hmin:.6f},  dt = {dt:.6f}")

In [ ]:
P1v = bufl.element("Lagrange", msh.basix_cell(), 1, shape=(2,))
P1s = bufl.element("Lagrange", msh.basix_cell(), 1)
V = functionspace(msh, P1v)
Q = functionspace(msh, P1s)

V0, _ = V.sub(0).collapse()
V1, _ = V.sub(1).collapse()

left_facets  = facet_tags.find(1)
right_facets = facet_tags.find(2)
lower_facets = facet_tags.find(3)
upper_facets = facet_tags.find(4)
obj_facets   = facet_tags.find(5)

def _bc_vel(val, sub_idx, sub_V, facets):
    f = Function(sub_V); f.x.array[:] = val
    dofs = fem.locate_dofs_topological((V.sub(sub_idx), sub_V), fdim, facets)
    return fem.dirichletbc(f, dofs, V.sub(sub_idx))

bcu_in0  = _bc_vel(uin, 0, V0, left_facets)
bcu_in1  = _bc_vel(0.0, 1, V1, left_facets)
bcu_upp1 = _bc_vel(0.0, 1, V1, upper_facets)
bcu_low1 = _bc_vel(0.0, 1, V1, lower_facets)
bcu_obj0 = _bc_vel(0.0, 0, V0, obj_facets)
bcu_obj1 = _bc_vel(0.0, 1, V1, obj_facets)
bcu = [bcu_in0, bcu_in1, bcu_upp1, bcu_low1, bcu_obj0, bcu_obj1]

bcp_out = fem.dirichletbc(
    Constant(msh, PETSc.ScalarType(0.0)),
    fem.locate_dofs_topological(Q, fdim, right_facets), Q)
bcp = [bcp_out]

print(f"V dofs: {V.dofmap.index_map.size_global * V.dofmap.index_map_bs}, "
      f"Q dofs: {Q.dofmap.index_map.size_global}")

In [ ]:
u0 = Function(V, name="u0"); u1 = Function(V, name="u1")
p1 = Function(Q, name="p1")

u = ufl.TrialFunction(V); v = ufl.TestFunction(V)
p = ufl.TrialFunction(Q); q = ufl.TestFunction(Q)

# GLS stabilization: factor 1/sqrt (NS convention, not 2/sqrt as in CD-NSE)
h_cell = ufl.CellDiameter(msh)
u_mag = ufl.sqrt(ufl.dot(u1, u1) + 1e-16)
d1 = 1.0 / ufl.sqrt((1.0/dt)**2 + (u_mag/h_cell)**2)
d2 = h_cell * u_mag  # bulk viscosity / div-div stabilization

# Trapezoidal means
um  = 0.5 * (u + u0)   # contains TrialFunction
um1 = 0.5 * (u1 + u0)  # known (linearization point)

# ── Momentum ──────────────────────────────────────────────────────────────
# Trapezoidal viscous: 0.5*nu on both sides
# d2 div-div stabilization: 0.5*d2 on both sides
au_expr = (
    ufl.inner(u/dt + 0.5*ufl.dot(ufl.grad(u), um1), v)
    + 0.5*nu * ufl.inner(ufl.grad(u), ufl.grad(v))
    + d1 * ufl.inner(u/dt + 0.5*ufl.dot(ufl.grad(u), um1),
                     ufl.dot(ufl.grad(v), um1))
    + 0.5*d2 * ufl.div(u) * ufl.div(v)
) * ufl.dx

Lu_expr = (
    ufl.inner(u0/dt - 0.5*ufl.dot(ufl.grad(u0), um1), v)
    + p1 * ufl.div(v)
    - 0.5*nu * ufl.inner(ufl.grad(u0), ufl.grad(v))
    + d1 * ufl.inner(u0/dt - 0.5*ufl.dot(ufl.grad(u0), um1) - ufl.grad(p1),
                     ufl.dot(ufl.grad(v), um1))
    - 0.5*d2 * ufl.div(u0) * ufl.div(v)
) * ufl.dx

# ── Pressure ──────────────────────────────────────────────────────────────
ap_expr = d1 * ufl.inner(ufl.grad(p), ufl.grad(q)) * ufl.dx
Lp_expr = (
    - d1 * ufl.inner((u1 - u0)/dt + ufl.dot(ufl.grad(um1), um1), ufl.grad(q))
    - ufl.div(um1) * q
) * ufl.dx

a_u = form(au_expr); l_u = form(Lu_expr)
a_p = form(ap_expr); l_p = form(Lp_expr)
print("Forms compiled.")

In [ ]:
A_u = create_matrix(a_u); b_u = create_vector(V)
A_p = create_matrix(a_p); b_p = create_vector(Q)

# Force on cylinder (stress integral, tag 5)
n_facet = ufl.FacetNormal(msh)
ds_msr = ufl.Measure("ds", domain=msh, subdomain_data=facet_tags)
sigma_u = -p1*ufl.Identity(2) + nu*(ufl.grad(u1) + ufl.grad(u1).T)
drag_form = form(ufl.dot(sigma_u * n_facet, ufl.as_vector([1.0, 0.0])) * ds_msr(5))
lift_form = form(ufl.dot(sigma_u * n_facet, ufl.as_vector([0.0, 1.0])) * ds_msr(5))

def make_ksp_bcgs_ilu(A):
    ksp = PETSc.KSP().create(msh.comm)
    ksp.setOperators(A)
    ksp.setType("bcgs")
    ksp.getPC().setType("ilu")
    ksp.setTolerances(rtol=1e-6, atol=1e-14, max_it=500)
    ksp.setFromOptions()
    return ksp

def make_ksp_bcgs_hypre(A):
    ksp = PETSc.KSP().create(msh.comm)
    ksp.setOperators(A)
    ksp.setType("bcgs")
    pc = ksp.getPC()
    pc.setType("hypre")
    pc.setHYPREType("boomeramg")
    ksp.setTolerances(rtol=1e-6, atol=1e-14, max_it=500)
    ksp.setFromOptions()
    return ksp

ksp_u = make_ksp_bcgs_ilu(A_u)
ksp_p = make_ksp_bcgs_hypre(A_p)
print("KSP solvers set up.")

In [ ]:
set_bc(u0.x.petsc_vec, bcu)
u0.x.scatter_forward()
set_bc(u1.x.petsc_vec, bcu)
u1.x.scatter_forward()

plot_time = 0.0
time_history = []
lift_history = []
drag_history = []
lift_dense_t = []
lift_dense = []

xdmf = XDMFSeries("results-nse/solution.xdmf")

t = dt
t_wall0 = time.time()
step = 0

while t < T + 1e-10:
    for _ in range(num_nnlin_iter):
        # Momentum
        A_u.zeroEntries()
        assemble_matrix(A_u, a_u, bcs=bcu)
        A_u.assemble()
        with b_u.localForm() as loc: loc.set(0.0)
        assemble_vector(b_u, l_u)
        apply_lifting(b_u, [a_u], bcs=[bcu])
        b_u.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b_u, bcu)
        ksp_u.setOperators(A_u, A_u)
        ksp_u.solve(b_u, u1.x.petsc_vec)
        u1.x.scatter_forward()
        set_bc(u1.x.petsc_vec, bcu)
        u1.x.scatter_forward()

        # Pressure
        A_p.zeroEntries()
        assemble_matrix(A_p, a_p, bcs=bcp)
        A_p.assemble()
        with b_p.localForm() as loc: loc.set(0.0)
        assemble_vector(b_p, l_p)
        apply_lifting(b_p, [a_p], bcs=[bcp])
        b_p.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b_p, bcp)
        ksp_p.setOperators(A_p, A_p)
        ksp_p.solve(b_p, p1.x.petsc_vec)
        p1.x.scatter_forward()
        set_bc(p1.x.petsc_vec, bcp)
        p1.x.scatter_forward()

    lift_dense.append(float(msh.comm.allreduce(assemble_scalar(lift_form), op=MPI.SUM)))
    lift_dense_t.append(t)

    if t > plot_time:
        drag_val = float(msh.comm.allreduce(assemble_scalar(drag_form), op=MPI.SUM))
        lift_val = lift_dense[-1]
        print(f"t = {t:.4f}  drag = {drag_val:.4f}  lift = {lift_val:.4f}")
        time_history.append(t)
        drag_history.append(drag_val)
        lift_history.append(lift_val)
        xdmf.write([u1, p1], t)
        plot_time += T / plot_freq

    u0.x.array[:] = u1.x.array
    t += dt
    step += 1

xdmf.close()
t_wall = time.time() - t_wall0
print(f"\nDone: {step} steps, wall time = {t_wall:.1f}s")

time_history = np.array(time_history)
lift_history = np.array(lift_history)
drag_history = np.array(drag_history)
lift_dense_t = np.array(lift_dense_t)
lift_dense   = np.array(lift_dense)

## Results

### Norms at final time

In [ ]:
from dolfinx.fem import assemble_scalar as _as

u_L2 = float(np.sqrt(_as(form(ufl.inner(u1, u1) * ufl.dx))))
p_L2 = float(np.sqrt(_as(form(p1**2 * ufl.dx))))
print(f"||u1||_L2 = {u_L2:.6f}")
print(f"||p1||_L2 = {p_L2:.6f}")
print(f"cells     = {n_cells}")
print(f"dt        = {dt:.6f}")
print(f"steps     = {step}")

In [ ]:
plot_vector(u1, title=f"Velocity at t={T:.0f}")
plot_scalar(p1, title=f"Pressure at t={T:.0f}")

In [ ]:
# Force history
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax1.plot(time_history, drag_history, label="Drag")
ax1.set_ylabel("Drag"); ax1.legend(); ax1.grid(True)
ax2.plot(time_history, lift_history, label="Lift", color="tab:orange")
ax2.set_ylabel("Lift"); ax2.set_xlabel("Time")
ax2.legend(); ax2.grid(True)
plt.suptitle("Force on cylinder (tag 5)")
plt.tight_layout(); plt.show()

# Strouhal number from dense lift history
D = 2.0 * cr
U = uin
if len(lift_dense_t) > 4 and lift_dense_t[-1] > 15.0:
    mask = lift_dense_t >= lift_dense_t[-1] - 15.0
    t_seg = lift_dense_t[mask]
    L_seg = lift_dense[mask]
    freqs = np.fft.rfftfreq(len(L_seg), d=dt)
    power = np.abs(np.fft.rfft(L_seg))**2
    f_dom = freqs[np.argmax(power[1:]) + 1]
    St = f_dom * D / U
    print(f"Dense lift samples (last 15 t.u.): {len(L_seg)}, dt = {dt:.4f}")
    print(f"Dominant lift frequency: {f_dom:.4f} Hz")
    print(f"Strouhal number St = f*D/U = {St:.4f}  (D={D}, U={U})")
    print(f"(Expected ~0.16 for Re=100 flow past circular cylinder)")
else:
    print("Lift history too short for Strouhal estimate (need T > 15).")

In [ ]:
# Triple decomposition: strain-rate S, spin W, Q-criterion
DG0 = functionspace(msh, ("DG", 0))
_pts = DG0.element.interpolation_points

_dux_dx = Function(DG0); _dux_dy = Function(DG0)
_duy_dx = Function(DG0); _duy_dy = Function(DG0)
_dux_dx.interpolate(fem.Expression(ufl.grad(u1)[0, 0], _pts))
_dux_dy.interpolate(fem.Expression(ufl.grad(u1)[0, 1], _pts))
_duy_dx.interpolate(fem.Expression(ufl.grad(u1)[1, 0], _pts))
_duy_dy.interpolate(fem.Expression(ufl.grad(u1)[1, 1], _pts))

nc = msh.topology.index_map(tdim).size_local
G00 = _dux_dx.x.array.real[:nc]; G01 = _dux_dy.x.array.real[:nc]
G10 = _duy_dx.x.array.real[:nc]; G11 = _duy_dy.x.array.real[:nc]

S01 = 0.5 * (G01 + G10)
Snorm2 = G00**2 + 2.0*S01**2 + G11**2
W01 = 0.5 * (G01 - G10)
Wnorm2 = 2.0 * W01**2
strain = np.sqrt(np.maximum(Snorm2, 0.0))
spin   = np.sqrt(Wnorm2)
Q_crit = Wnorm2 - Snorm2

x = msh.geometry.x
gdm = msh.geometry.dofmaps[0]
fig, axes = plt.subplots(3, 1, figsize=(10, 9))
for ax, vals, title in zip(axes,
    [strain, spin, Q_crit],
    [f"|S| strain rate at t={T:.0f}", f"|W| spin at t={T:.0f}",
     f"Q-criterion at t={T:.0f}"]):
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="flat")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Discussion

The incompressible Navier–Stokes equations are solved by a linearized
fractional-step scheme: at each nonlinear iteration the momentum equation is
solved for $u$ with frozen pressure $p_1$, then the continuity projection
updates $p$.  GLS stabilization with $\delta_1 = 1/\sqrt{(1/\Delta t)^2 + (|u|/h)^2}$
and the bulk-viscosity term $\delta_2 = h|u|$ (divergence stabilization)
enable stable equal-order $P_1/P_1$ elements.  The viscous term is treated
by the trapezoidal rule: $\nu\,\nabla\bar{u}\cdot\nabla v$ with $\bar{u} = \tfrac12(u^{n}+u^{n-1})$.

The Strouhal number $\text{St} = fD/U$ ($D=0.4$, $U=1$) characterizes
vortex-shedding frequency behind the cylinder.  For $\text{Re}=100$ the
expected value is $\text{St} \approx 0.16$.

The triple decomposition separates the velocity gradient $\nabla u = S + W$ into
symmetric strain-rate $S$ and antisymmetric spin $W$.  The Q-criterion $Q = |W|^2 - |S|^2 > 0$
identifies rotation-dominated regions (vortex cores) in the wake.

XDMF output is written to `results-nse/solution.xdmf` for ParaView
visualization.